<div dir="rtl">

# تحليل طلبات المطعم — تقرير الإجابات

يجيب هذا الدفتر عن أسئلة الأداء والازدحام والقنوات والأصناف والتوصيل، ثم يعرض ثلاث توصيات مبنية على نتائج البيانات. جميع الإيرادات بالدولار.

**مصدر البيانات:** `restaurant_orders.csv` — طلبات يناير إلى يونيو 2025.

> قبل التحليل، يزيل الدفتر الصفوف المتطابقة بالكامل ويوحّد أسماء الفروع. لا يغيّر ملف CSV الأصلي. يُحسب عدد الطلبات من عدد أرقام الطلبات الفريدة، ومتوسط قيمة الطلب = الإيراد ÷ عدد الطلبات.

</div>

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

data_path = Path("restaurant_orders.csv")
if not data_path.exists():
    data_path = Path("..") / ".." / "restaurant_orders.csv"
if not data_path.exists():
    raise FileNotFoundError("لم أجد restaurant_orders.csv. ضع ملف البيانات بجانب هذا الدفتر أو عدّل data_path.")

raw = pd.read_csv(data_path)
duplicate_count = int(raw.duplicated().sum())
df = raw.drop_duplicates().copy()

branch_names = {
    "downtown": "Downtown",
    "mall": "Mall",
    "university": "University",
}
df["branch"] = df["branch"].astype("string").str.strip().str.casefold().map(branch_names)
if df["branch"].isna().any():
    unexpected = raw.loc[df["branch"].isna(), "branch"].dropna().unique().tolist()
    raise ValueError(f"توجد أسماء فروع غير معروفة وتحتاج إلى مراجعة: {unexpected}")

df["order_date"] = pd.to_datetime(df["order_date"], errors="raise")
for column in ["order_hour", "quantity", "unit_price", "total", "delivery_minutes", "rating"]:
    df[column] = pd.to_numeric(df[column], errors="raise")

weekday_ar = {
    0: "الاثنين", 1: "الثلاثاء", 2: "الأربعاء", 3: "الخميس",
    4: "الجمعة", 5: "السبت", 6: "الأحد",
}
df["day_name"] = df["order_date"].dt.dayofweek.map(weekday_ar)

display(Markdown(
    f"**حجم البيانات الأصلي:** {len(raw):,} صفًا · "
    f"**صفوف مكررة أزيلت للتحليل:** {duplicate_count:,} · "
    f"**الصفوف بعد إزالة التكرار:** {len(df):,} · "
    f"**عدد الطلبات الفريدة:** {df['order_id'].nunique():,}"
))

: 

<div dir="rtl">

## السؤال 1: ما ساعات الذروة؟ وأي أيام الأسبوع أكثر ازدحامًا؟

الترتيب تنازلي حسب عدد الطلبات الفريدة. ساعة الذروة واليوم الأكثر ازدحامًا هما أعلى نتيجة في كل جدول.

</div>

In [ ]:
hourly = (
    df.groupby("order_hour")["order_id"]
      .nunique()
      .sort_values(ascending=False)
      .rename("عدد الطلبات")
      .rename_axis("ساعة الطلب")
      .reset_index()
)
daily = (
    df.groupby("day_name")["order_id"]
      .nunique()
      .sort_values(ascending=False)
      .rename("عدد الطلبات")
      .rename_axis("اليوم")
      .reset_index()
)
peak_hour = int(hourly.iloc[0]["ساعة الطلب"])
peak_hour_count = int(hourly.iloc[0]["عدد الطلبات"])
busiest_day = str(daily.iloc[0]["اليوم"])
busiest_day_count = int(daily.iloc[0]["عدد الطلبات"])

display(Markdown(f"**الإجابة:** ساعة الذروة هي **{peak_hour}:00** بـ **{peak_hour_count:,} طلبًا**. وأكثر الأيام ازدحامًا هو **{busiest_day}** بـ **{busiest_day_count:,} طلبًا**."))
display(Markdown("**الطلبات حسب الساعة**"))
display(hourly)
display(Markdown("**الطلبات حسب يوم الأسبوع**"))
display(daily)

<div dir="rtl">

## السؤال 2: ما الإيرادات وعدد الطلبات ومتوسط قيمة الطلب لكل فرع؟ وأي فرع الأفضل؟

نعرّف «الأفضل» هنا بأنه الفرع الأعلى **إجماليًا في الإيرادات** خلال فترة البيانات. الجدول يعرض أيضًا عدد الطلبات ومتوسط قيمة الطلب للمقارنة.

</div>

In [ ]:
branch_summary = df.groupby("branch").agg(
    الإيرادات=("total", "sum"),
    عدد_الطلبات=("order_id", "nunique"),
).sort_values("الإيرادات", ascending=False)
branch_summary["متوسط قيمة الطلب"] = branch_summary["الإيرادات"] / branch_summary["عدد_الطلبات"]
branch_table = branch_summary.rename_axis("الفرع").reset_index()
best_branch = str(branch_summary.index[0])
best_branch_revenue = float(branch_summary.iloc[0]["الإيرادات"])
display(Markdown(f"**الإجابة:** الأفضل من حيث إجمالي الإيرادات هو فرع **{best_branch}** بإيرادات **${best_branch_revenue:,.2f}**."))
display(branch_table.style.format({"الإيرادات": "${:,.2f}", "متوسط قيمة الطلب": "${:,.2f}", "عدد_الطلبات": "{:,.0f}"}).hide(axis="index"))

<div dir="rtl">

## السؤال 3: ما حصة كل قناة من الإيرادات؟

القنوات: صالة (`Dine-in`)، سفري (`Takeaway`)، وتوصيل (`Delivery`). تُحسب الحصة من إجمالي الإيرادات بعد إزالة التكرارات.

</div>

In [ ]:
channel_ar = {"Dine-in": "صالة", "Takeaway": "سفري", "Delivery": "توصيل"}
channel_summary = df.groupby("channel")["total"].sum().sort_values(ascending=False).to_frame("الإيرادات")
channel_summary["الحصة من الإيرادات"] = channel_summary["الإيرادات"] / df["total"].sum() * 100
channel_summary = channel_summary.rename(index=channel_ar).rename_axis("القناة").reset_index()
top_channel = str(channel_summary.iloc[0]["القناة"])
top_channel_share = float(channel_summary.iloc[0]["الحصة من الإيرادات"])
display(Markdown(f"**الإجابة:** أكبر مساهم في الإيرادات هو **{top_channel}** بحصة **{top_channel_share:.1f}%**."))
display(channel_summary.style.format({"الإيرادات": "${:,.2f}", "الحصة من الإيرادات": "{:.1f}%"}).hide(axis="index"))

<div dir="rtl">

## السؤال 4: ما أعلى خمسة أصناف من حيث الإيرادات؟

</div>

In [ ]:
top_items = (
    df.groupby("item")["total"].sum()
      .sort_values(ascending=False)
      .head(5)
      .rename("الإيرادات")
      .rename_axis("الصنف")
      .reset_index()
)
top_item_name = str(top_items.iloc[0]["الصنف"])
top_item_revenue = float(top_items.iloc[0]["الإيرادات"])
display(Markdown(f"**الإجابة:** الصنف الأعلى إيرادًا هو **{top_item_name}** بإيرادات **${top_item_revenue:,.2f}**. وفيما يلي أعلى خمسة أصناف:"))
display(top_items.style.format({"الإيرادات": "${:,.2f}"}).hide(axis="index"))

<div dir="rtl">

## السؤال 5: هل يرتبط وقت التوصيل بتقييم العميل؟

نقارن متوسط التقييم ضمن ثلاث فئات: أقل من 30 دقيقة، من 30 إلى 45 دقيقة **شاملًا الحدّين**، وأكثر من 45 دقيقة. يُحسب المتوسط من الطلبات التي لديها تقييم فقط، مع عرض عدد الطلبات وعدد التقييمات المحتسبة.

</div>

In [ ]:
delivery = df.loc[df["channel"].eq("Delivery") & df["delivery_minutes"].notna()].copy()
delivery["فئة وقت التوصيل"] = pd.Series(index=delivery.index, dtype="string")
delivery.loc[delivery["delivery_minutes"] < 30, "فئة وقت التوصيل"] = "أقل من 30 دقيقة"
delivery.loc[delivery["delivery_minutes"].between(30, 45, inclusive="both"), "فئة وقت التوصيل"] = "من 30 إلى 45 دقيقة"
delivery.loc[delivery["delivery_minutes"] > 45, "فئة وقت التوصيل"] = "أكثر من 45 دقيقة"
delivery_order = ["أقل من 30 دقيقة", "من 30 إلى 45 دقيقة", "أكثر من 45 دقيقة"]
delivery["فئة وقت التوصيل"] = pd.Categorical(delivery["فئة وقت التوصيل"], categories=delivery_order, ordered=True)
delivery_summary = delivery.groupby("فئة وقت التوصيل", observed=False).agg(
    عدد_الطلبات=("order_id", "nunique"),
    عدد_التقييمات=("rating", "count"),
    متوسط_التقييم=("rating", "mean"),
).reset_index()
delivery_summary = delivery_summary.rename(columns={
    "عدد_الطلبات": "عدد الطلبات",
    "عدد_التقييمات": "عدد التقييمات المحتسبة",
    "متوسط_التقييم": "متوسط التقييم",
})
rated_means = delivery_summary.dropna(subset=["متوسط التقييم"])
if len(rated_means) >= 2:
    quickest_mean = float(rated_means.iloc[0]["متوسط التقييم"])
    slowest_mean = float(rated_means.iloc[-1]["متوسط التقييم"])
    relation_text = (
        f"متوسط التقييم في فئة التوصيل الأسرع ({quickest_mean:.2f}) أعلى من الفئة الأبطأ ({slowest_mean:.2f})."
        if quickest_mean > slowest_mean else
        f"متوسط التقييم في فئة التوصيل الأسرع ({quickest_mean:.2f}) ليس أعلى من الفئة الأبطأ ({slowest_mean:.2f})."
    )
    relation_text += " هذه مقارنة وصفية ولا تثبت أن مدة التوصيل هي سبب اختلاف التقييم."
else:
    relation_text = "لا توجد تقييمات كافية في فئات متعددة لإجراء مقارنة موثوقة."
display(Markdown(f"**الإجابة:** {relation_text}"))
display(delivery_summary.style.format({"متوسط التقييم": "{:.2f}"}, na_rep="لا توجد تقييمات").hide(axis="index"))

<div dir="rtl">

## السؤال 6: ثلاث توصيات لإدارة المطعم

التوصيات التالية تُنشأ من الجداول أعلاه: تركز على وقت الذروة، والقناة والأصناف الأعلى إيرادًا، وتجربة التوصيل. الارتباط في البيانات لا يثبت السببية.

</div>

In [ ]:
delivery_rows = delivery_summary.set_index("فئة وقت التوصيل")
fast_avg = delivery_rows.loc["أقل من 30 دقيقة", "متوسط التقييم"]
slow_avg = delivery_rows.loc["أكثر من 45 دقيقة", "متوسط التقييم"]
if pd.notna(fast_avg) and pd.notna(slow_avg):
    delivery_evidence = (
        f"متوسط التقييم أقل من 30 دقيقة {fast_avg:.2f} مقابل {slow_avg:.2f} لأكثر من 45 دقيقة "
        f"(عدد التقييمات: {int(delivery_rows.loc['أقل من 30 دقيقة', 'عدد التقييمات المحتسبة'])} و"
        f"{int(delivery_rows.loc['أكثر من 45 دقيقة', 'عدد التقييمات المحتسبة'])} على الترتيب)."
    )
else:
    delivery_evidence = "لا تتوفر تقييمات كافية للمقارنة بين أسرع وأبطأ فئة؛ اجمع بيانات إضافية قبل اتخاذ قرار تشغيلي."

recommendations = [
    (
        "توزيع الموظفين والمخزون حسب الذروة",
        f"عزّز التغطية في الساعة {peak_hour}:00 ويوم {busiest_day}، وراجع الجدولة أسبوعيًا.",
        f"هما الأعلى في حجم الطلبات: {peak_hour_count:,} طلبًا في ساعة الذروة و{busiest_day_count:,} في اليوم الأكثر ازدحامًا."
    ),
    (
        "دعم القناة والأصناف الأعلى إيرادًا",
        f"حافظ على توفر الأصناف الأعلى مبيعًا، ووجّه العروض إلى القناة {top_channel} مع متابعة هامش الربح قبل توسيع الخصومات.",
        f"القناة {top_channel} تمثل {top_channel_share:.1f}% من الإيرادات، والصنف {top_item_name} هو الأعلى بإيرادات ${top_item_revenue:,.2f}."
    ),
    (
        "متابعة سرعة التوصيل ورضا العملاء",
        "راجع أسباب التأخير في الطلبات التي تتجاوز 45 دقيقة، وتابع متوسط التقييم أسبوعيًا قبل وبعد أي تحسين.",
        delivery_evidence
    ),
]
recommendation_table = pd.DataFrame(recommendations, columns=["التوصية", "الإجراء المقترح", "الدليل من البيانات"])
display(recommendation_table.style.hide(axis="index"))